# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Samia2310/flyrank-assignment1-week1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Two Paper Findings + My Methodology Questions

### Finding 1

The research paper reports that ensemble learning methods achieved stronger fraud detection performance than traditional machine learning models.

**Methodology Question**

How was the fraud label created and verified? Understanding the source and quality of the labels is important because model performance depends on the reliability of the ground truth.

---

### Finding 2

The paper reports that Random Forest and AdaBoost achieved the highest evaluation scores.

**Methodology Question**

What validation strategy was used to evaluate the models? A grouped or time-aware validation design generally provides a more realistic estimate of future performance than a simple random split.

These questions are intended to better understand the methodology rather than challenge the reported results.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## My Model Under an Honest Split

In Week 5, the Random Forest model was evaluated using a random train-test split.

For this validation audit, the model is evaluated using a grouped split based on `client_id`. Grouping prevents information from the same client appearing in both the training and testing sets, providing a more realistic estimate of performance.

This comparison helps determine whether the original evaluation may have been optimistic.

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("content_refresh_anonymized.csv")

# Recreate Week-4 labels (business rule, not model-derived)
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["low_ctr"] = (df["ctr"] < 1.0).astype(int)

df["baseline_score"] = df["stale"]*2 + df["visible"]*2 + df["low_ctr"]

def action(score):
    if score >= 5:
        return "Refresh Immediately"
    elif score >= 3:
        return "Review Soon"
    else:
        return "Monitor"

df["action"] = df["baseline_score"].apply(action)

# --- BEFORE ---
# These three columns were used to BUILD the label above, so keeping them in X
# lets the model recover the label-generating rule instead of learning a real
# pattern. Computed here on purpose to show why this number can't be trusted.
leaky_features = ["days_since_last_update", "impressions_90d", "ctr"]
other_features = ["content_age_days", "search_volume", "avg_position",
                   "engagement_rate", "scroll_rate"]

X_leaky = df[leaky_features + other_features].fillna(0)
y = df["action"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_leaky, y, test_size=0.20, random_state=42, stratify=y
)

rf_before = RandomForestClassifier(n_estimators=200, random_state=42)
rf_before.fit(X_train_r, y_train_r)
pred_before = rf_before.predict(X_test_r)
accuracy_before = accuracy_score(y_test_r, pred_before)

print("BEFORE — random split, leaky features included")
print("Accuracy:", round(accuracy_before, 4))

BEFORE — random split, leaky features included
Accuracy: 0.9998


In [10]:
# --- AFTER ---
# Fix 1: drop the features used to construct the label
# Fix 2: use a grouped split by client_id instead of a random split
X_honest = df[other_features].fillna(0)
groups = df["client_id"]

splitter = GroupShuffleSplit(test_size=0.20, random_state=42, n_splits=1)
train_idx, test_idx = next(splitter.split(X_honest, y, groups))

X_train, X_test = X_honest.iloc[train_idx], X_honest.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
accuracy_after = accuracy_score(y_test, pred)

print("AFTER — grouped split, leaky features removed")
print("Accuracy:", round(accuracy_after, 4))

AFTER — grouped split, leaky features removed
Accuracy: 0.6633


In [11]:
comparison = pd.DataFrame({
    "Validation": [
        "Random split + leaky features (Week 5 style)",
        "Grouped split (by client_id) + leaky features removed"
    ],
    "Accuracy": [round(accuracy_before, 4), round(accuracy_after, 4)]
})
comparison

,Validation,Accuracy
0,Random split + leaky features (Week 5 style),0.9998
1,Grouped split (by client_id) + leaky features ...,0.6633


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



Only historical information available at prediction time is used for model training.

The following columns were intentionally excluded because they may introduce leakage or contain future-derived information:

- trend_direction
- trend_pct

The model uses historical search performance, content freshness, search visibility, and engagement features only.

In [12]:
used_features = other_features

excluded_features = [
    "days_since_last_update",  # used to build the "stale" part of the label
    "impressions_90d",         # used to build the "visible" part of the label
    "ctr",                     # used to build the "low_ctr" part of the label
    "trend_direction",         # future-derived trend signal
    "trend_pct"                # future-derived trend signal
]

print("Features used")
print(used_features)
print()
print("Leakage features excluded")
print(excluded_features)
print()
print("Reason: the first three were used to directly construct the target")
print("label ('action'), so including them let the model recover the label-")
print("generating rule instead of learning a genuine pattern. The last two")
print("are future-derived and not available at prediction time.")

Features used
['content_age_days', 'search_volume', 'avg_position', 'engagement_rate', 'scroll_rate']

Leakage features excluded
['days_since_last_update', 'impressions_90d', 'ctr', 'trend_direction', 'trend_pct']

Reason: the first three were used to directly construct the target
label ('action'), so including them let the model recover the label-
generating rule instead of learning a genuine pattern. The last two
are future-derived and not available at prediction time.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



### Original Claim

The Random Forest model accurately identifies which pages should be refreshed.

### Revised Claim

The Random Forest model showed improved predictive performance on the available historical dataset under the evaluated validation design. The model provides decision-support for identifying pages that may benefit from review, but its recommendations should be interpreted alongside human judgment.

This wording reflects the observed evidence without making broader claims than the evaluation supports.

In [13]:
errors = pd.DataFrame({"Actual": y_test, "Predicted": pred})
errors = errors[errors["Actual"] != errors["Predicted"]]

print(f"Misclassified: {len(errors)} of {len(y_test)} test rows")
errors.head(10)

Misclassified: 2075 of 6163 test rows


,Actual,Predicted
5,Review Soon,Monitor
22,Review Soon,Monitor
43,Monitor,Review Soon
56,Monitor,Review Soon
64,Review Soon,Monitor
82,Review Soon,Monitor
90,Monitor,Review Soon
126,Monitor,Review Soon
127,Review Soon,Monitor
129,Review Soon,Monitor


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.